In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_ciclo = spark.table(f"{catalogo}.{esquema_source}.ciclo")
df_programa = spark.table(f"{catalogo}.{esquema_source}.programa")
df_matricula = spark.table(f"{catalogo}.{esquema_source}.matricula")

In [0]:
df_matricula_ue_select = df_matricula.select("ACAD_CAREER",
                                          "STRM",
                                          "EMPLID",
                                          "EMPLID_NAME",
                                          "ACAD_PROG",
                                          "CYCLE",
                                          "CAMPUS"
                                          )

In [0]:
df_matricula_ue_distinct = df_matricula_ue_select.distinct()

In [0]:
df_matricula_ue_cycle_without_c = df_matricula_ue_distinct.withColumn("CYCLE", regexp_replace(col("CYCLE"), "C", ""))

In [0]:
df_matricula_ue_cycle_number = df_matricula_ue_cycle_without_c.withColumn("CYCLE", col("CYCLE").cast('int'))

In [0]:
df_matricula_ue = df_matricula_ue_cycle_number.groupBy("ACAD_CAREER",
                                                 "STRM",
                                                 "EMPLID",
                                                 "EMPLID_NAME",
                                                 "ACAD_PROG",
                                                 "CAMPUS").agg(F.max("CYCLE").alias("CYCLE"))

In [0]:
df_escuela = df_programa.select("ACAD_PROG",
                                "ACAD_GROUP"
                                )

In [0]:
df_joined_matriculados_escuela = df_matricula_ue.alias("m").join(df_escuela.alias("e"), col("m.ACAD_PROG") == col("e.ACAD_PROG"), "inner")\
                                                           .join(df_ciclo.alias("c"),col("c.STRM") == col("m.STRM"), "inner")

In [0]:
df_joined_ue_select = df_joined_matriculados_escuela.select(col("m.ACAD_CAREER"),
                                                            col("m.STRM"), 
                                                            col("c.DESCR"),
                                                            col("m.EMPLID"), 
                                                            col("m.EMPLID_NAME"),
                                                            col("m.ACAD_PROG"),
                                                            col("e.ACAD_GROUP")).orderBy("m.EMPLID")


In [0]:
df_joined_ue_filter = df_joined_ue_select.filter(col("ACAD_GROUP") == "SALUD")

In [0]:
df_joined_ue_filter.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.matriculados_unicos_escuela")